In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedGroupKFold, learning_curve
from sklearn.metrics import make_scorer, f1_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.under_sampling import RandomUnderSampler

import plotly.graph_objects as go
import numpy as np


In [13]:
df_train = pd.read_csv('./Datos/df_voices_train.csv')

<>:1: SyntaxWarning: invalid escape sequence '\d'
<>:1: SyntaxWarning: invalid escape sequence '\d'
C:\Users\SOTEC\AppData\Local\Temp\ipykernel_28492\3029133970.py:1: SyntaxWarning: invalid escape sequence '\d'
  df_train = pd.read_csv('./Datos\df_general.csv')


In [3]:
# Variable objetivo
df_train['y'] = df_train['Key'].map({
    'bonafide': 0,
    'spoof': 1
})

# Variables categóricas
if 'Gender' in df_train.columns:
    df_train = pd.get_dummies(
        df_train,
        columns=['Gender'],
        drop_first=True
    )

In [4]:
# Separar variables
drop_columns = ['Key', 'file_name', 'User_ID', 'Spoofing_ID', 'y']

X = df_train.drop(columns=drop_columns, errors='ignore')
y = df_train['y']

# Groups para evitar leakage
groups = df_train['Spoofing_ID']

print(f"Muestras: {X.shape[0]}")
print(f"Variables: {X.shape[1]}\n")


Muestras: 25380
Variables: 22



In [5]:
# ==========================================
# 1. MODELO
# ==========================================

pipeline = Pipeline([
    ('scaler', StandardScaler()),

    ('model', RandomForestClassifier(
        n_estimators=250,
        random_state=42,
        n_jobs=-1,
        class_weight='balanced'
    ))
])

In [6]:
# ==========================================
# 2. CROSS VALIDATION
# ==========================================

cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [7]:
# ==========================================
# 3. LEARNING CURVE
# ==========================================

print("--- CALCULANDO LEARNING CURVE ---")

train_sizes, train_scores, validation_scores = learning_curve(
    estimator=pipeline,
    X=X,
    y=y,
    groups=groups,
    cv=cv,

    scoring=make_scorer(f1_score),

    # tamaños ABSOLUTOS
    train_sizes=np.arange(500, 10001, 500),

    n_jobs=-1,

    shuffle=True,
    random_state=42
)

# Promedios
train_mean = np.mean(train_scores, axis=1)
train_std = np.std(train_scores, axis=1)

validation_mean = np.mean(validation_scores, axis=1)
validation_std = np.std(validation_scores, axis=1)

--- CALCULANDO LEARNING CURVE ---


In [8]:
results_df = pd.DataFrame({
    'Muestras entrenamiento': train_sizes,
    'F1 Train': train_mean,
    'F1 Validation': validation_mean
})

print("\n==========================================")
print("RESULTADOS LEARNING CURVE")
print("==========================================\n")

print(results_df)



RESULTADOS LEARNING CURVE

    Muestras entrenamiento  F1 Train  F1 Validation
0                      500       1.0       0.928960
1                     1000       1.0       0.922526
2                     1500       1.0       0.914210
3                     2000       1.0       0.912299
4                     2500       1.0       0.912419
5                     3000       1.0       0.912849
6                     3500       1.0       0.912743
7                     4000       1.0       0.908486
8                     4500       1.0       0.907868
9                     5000       1.0       0.907411
10                    5500       1.0       0.906555
11                    6000       1.0       0.907490
12                    6500       1.0       0.907934
13                    7000       1.0       0.906249
14                    7500       1.0       0.906962
15                    8000       1.0       0.906922
16                    8500       1.0       0.905143
17                    9000       1.0

In [ ]:
fig = go.Figure()

# 1. Sombra de desviación estándar para Entrenamiento
fig.add_trace(go.Scatter(
    x=np.concatenate([train_sizes, train_sizes[::-1]]),
    y=np.concatenate([train_mean + train_std, (train_mean - train_std)[::-1]]),
    fill='toself',
    fillcolor='rgba(31, 59, 115, 0.15)',
    line=dict(color='rgba(255,255,255,0)'),
    hoverinfo="skip",
    showlegend=False
))

# 2. Sombra de desviación estándar para Validación
fig.add_trace(go.Scatter(
    x=np.concatenate([train_sizes, train_sizes[::-1]]),
    y=np.concatenate([validation_mean + validation_std, (validation_mean - validation_std)[::-1]]),
    fill='toself',
    fillcolor='rgba(122, 31, 31, 0.15)',
    line=dict(color='rgba(255,255,255,0)'),
    hoverinfo="skip",
    showlegend=False
))

# 3. Línea de Entrenamiento (F1 Train)
fig.add_trace(go.Scatter(
    x=train_sizes, 
    y=train_mean,
    mode='lines+markers',
    name='F1 Train',
    line=dict(color='#1F3B73', width=3),
    marker=dict(size=8)
))

# 4. Línea de Validación (F1 Validation)
fig.add_trace(go.Scatter(
    x=train_sizes, 
    y=validation_mean,
    mode='lines+markers',
    name='F1 Validation',
    line=dict(color='#7A1F1F', width=3),
    marker=dict(size=8)
))

# 5. Configuración de diseño (Estética Premium)
fig.update_layout(
    title={
        'text': "<b>Learning Curve — Random Forest (250 árboles)</b>",
        'font': {'size': 22, 'color': '#333'},
        'x': 0.5, 'xanchor': 'center'
    },
    xaxis=dict(
        title='Número de muestras de entrenamiento',
        gridcolor='#f0f0f0',
        zeroline=False,
                tickformat='d'  
    ),
    yaxis=dict(
        title='F1-Score',
        gridcolor='#f0f0f0',
        zeroline=False,
        range=[0.85, 1.01] 
    ),
    template='plotly_white',
    hovermode='x unified',
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1,
        bordercolor="#ddd",
        borderwidth=1
    ),
    margin=dict(l=60, r=40, t=100, b=60),
    height=600
)

fig.show()
